# 02 — ANP: tratamento e doses

Constrói a tabela municipal de tratamento a partir dos 3 XLSXs de certificados ANP, integrando NEEA e volume elegível como doses contínuas T2 e T3.

**Filtro metodológico:** Etanol Anidro × cana 1G (decisão D4 — pré-registro v2.2 §3.5).

**Outputs em `data/interim/`:**
- `anp_eventos_raw.csv` — universo bruto (todas rotas, todos status, 3 snapshots)
- `anp_eventos_anidro_cana1g.csv` — filtrado para Anidro × cana 1G válido
- `anp_first_cert.csv` — 1ª certificação por CNPJ × cana 1G
- `anp_neea_anidro_panel.csv` — NEEA por usina × snapshot
- `anp_muni_treat.csv` — tabela municipal (input do PSM)

**Auditorias em `outputs_pre/`:**
- `anp_zero_neea_audit.csv` — usinas com NEEA=0 em algum snapshot
- `anp_cancelados_audit.csv` — usinas em Cancelados/Anulados
- `anp_cidade_uf_unmatched.csv` — emissores sem match no crosswalk

In [1]:
# Setup: monta Drive, adiciona pipeline ao sys.path
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/Renovabio - EcoEco')
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Reload módulos (importante após editar os .py)
import importlib
from pipeline import config, normalize, io, crosswalk, anp
importlib.reload(config)
importlib.reload(normalize)
importlib.reload(io)
importlib.reload(crosswalk)
importlib.reload(anp)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.anp import run_anp_pipeline
print('✓ módulos carregados')

✓ módulos carregados


In [3]:
# Carrega o crosswalk universal (gerado no notebook 01)
cw_path = interim('crosswalk_centrosul.csv')
if not cw_path.exists():
    raise FileNotFoundError(
        f'Crosswalk não encontrado em {cw_path}. '
        f'Rode o notebook 01_crosswalk.ipynb antes deste.'
    )

cw = pd.read_csv(cw_path, dtype={'geocode': str})
print(f'Crosswalk: {cw.shape}')
print(f'Colunas: {list(cw.columns)}')
print(f'\nAmostra:')
cw.head(3)

Crosswalk: (2363, 5)
Colunas: ['geocode', 'municipio', 'uf', 'cidade_uf_seeg', 'muni_key']

Amostra:


,geocode,municipio,uf,cidade_uf_seeg,muni_key
0,5200050,Abadia de Goiás,GO,Abadia de Goiás (GO),ABADIA DE GOIAS|GO
1,5200100,Abadiânia,GO,Abadiânia (GO),ABADIANIA|GO
2,5200134,Acreúna,GO,Acreúna (GO),ACREUNA|GO


## Rodar pipeline ANP completo

Lê 3 snapshots, faz parsing + harmonização, gera 5 tabelas interim e 3 auditorias, salva tudo em disco.

In [4]:
# Roda o pipeline (save=True salva em interim/ e outputs_pre/)
result = run_anp_pipeline(cw, save=True)

→ Lendo 3 snapshots ANP com unmerge+fill...
  eventos_raw: 3,025 linhas
  eventos_anidro_cana1g: 526 linhas, 204 CNPJs únicos
  first_cert: 298 CNPJs cana 1G
  neea_panel: 525 linhas (CNPJ × snapshot)
  audit_zero_neea: 34 CNPJs com algum NEEA ausente/zero
  audit_cancelados: 283 CNPJs com cancelado/anulado
  muni_treat: 194 municípios tratados (Centro-Sul)
  audit_unmatched: 65 CNPJs sem geocode

→ Salvando interim/ e outputs_pre/...
  ✓ todos os arquivos salvos


## Inspeção do muni_treat

In [5]:
# Tabela municipal — input principal do PSM/CS/SDID
muni = result['muni_treat']
print(f'Shape: {muni.shape}')
print(f'\nColunas:')
for c in muni.columns:
    print(f'  - {c}')
muni.head()

Shape: (194, 13)

Colunas:
  - geocode
  - municipio
  - uf
  - g_m
  - g_data_m
  - n_usinas
  - n_usinas_baseline
  - dose_T2_2022
  - dose_T2_2025
  - dose_T2_2026
  - dose_T3_2022
  - dose_T3_2025
  - dose_T3_2026


,geocode,municipio,uf,g_m,g_data_m,n_usinas,n_usinas_baseline,dose_T2_2022,dose_T2_2025,dose_T2_2026,dose_T3_2022,dose_T3_2025,dose_T3_2026
0,5201306,Anicuns,GO,2020,2020-06-12,1,0,100.00,79.23,79.22,61.2,49.12,57.50
1,5204250,Cachoeira Dourada,GO,2020,2020-03-30,1,0,98.70,98.48,89.80,58.7,58.23,61.48
2,5205000,Carmo do Rio Verde,GO,2020,2020-04-06,1,0,86.86,96.81,NaN,64.0,67.37,NaN
3,5204300,Caçu,GO,2020,2020-02-11,1,0,95.44,98.33,94.04,58.6,62.40,61.47
4,5205471,Chapadão do Céu,GO,2020,2020-01-16,1,0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Distribuição por UF
print('Por UF:')
print(muni.groupby('uf').size().to_string())
print(f'\nTotal municípios tratados: {len(muni)}')

print('\n\nDistribuição da coorte g_m:')
print(muni['g_m'].value_counts().sort_index().to_string())

print('\n\nn_usinas_baseline (até 2019):')
print(muni['n_usinas_baseline'].value_counts().sort_index().to_string())
n_baseline = (muni['n_usinas_baseline'] > 0).sum()
print(f'  N municípios com baseline > 0: {n_baseline}')

Por UF:
uf
GO     26
MG     25
MS     15
MT      7
PR     17
SP    104

Total municípios tratados: 194


Distribuição da coorte g_m:
g_m
2019      2
2020    142
2021     40
2022      8
2023      2


n_usinas_baseline (até 2019):
n_usinas_baseline
0    192
1      2
  N municípios com baseline > 0: 2


In [7]:
# Doses T2 e T3 por snapshot
print('Doses T2 (volume elegível, %) e T3 (NEEA, gCO₂eq/MJ):\n')
for snap in ['2022', '2025', '2026']:
    for tipo in ['T2', 'T3']:
        col = f'dose_{tipo}_{snap}'
        if col in muni.columns:
            s = muni[col].dropna()
            print(f'  {col}: n={len(s)}, mean={s.mean():.2f}, '
                  f'range=[{s.min():.2f}, {s.max():.2f}]')
    print()

Doses T2 (volume elegível, %) e T3 (NEEA, gCO₂eq/MJ):

  dose_T2_2022: n=116, mean=93.68, range=[16.01, 100.00]
  dose_T3_2022: n=116, mean=60.31, range=[48.30, 70.50]

  dose_T2_2025: n=124, mean=92.66, range=[0.82, 100.00]
  dose_T3_2025: n=124, mean=60.28, range=[41.69, 71.29]

  dose_T2_2026: n=113, mean=91.43, range=[58.06, 100.00]
  dose_T3_2026: n=113, mean=61.56, range=[48.73, 70.79]



## Auditorias

Três tabelas em `outputs_pre/` para revisão visual.

In [8]:
# Auditoria de NEEA=0 / ausências
audit_z = result['audit_zero_neea']
print(f'NEEA=0 audit: {len(audit_z)} CNPJs\n')
print('Padrões:')
print(audit_z['padrao'].value_counts().to_string())
print()
audit_z.head(10)

NEEA=0 audit: 34 CNPJs

Padrões:
padrao
adocao_tardia    34



,cnpj_clean,2022,2025,2026,padrao,razao_social,cidade,uf
125,30974737000176,NaN,56.77,57.520,adocao_tardia,ALCON,Conceição da Barra,ES
8,02460988000105,NaN,62.45,62.920,adocao_tardia,USINA GOIANÉSIA S/A,Goianésia,GO
89,11092881000134,NaN,55.73,55.730,adocao_tardia,Bom Sucesso Agroindustria S.A.,Goiatuba,GO
122,28144326000101,NaN,56.15,56.850,adocao_tardia,CANÁPOLIS AÇÚCAR E ETANOL S.A.,Canápolis,MG
68,08793343000162,NaN,65.38,65.915,adocao_tardia,BIOENERGÉTICA VALE DO PARACATU S/A,João Pinheiro,MG
115,18054379000188,NaN,48.03,48.380,adocao_tardia,DASA - DESTILARIA DE ÁLCOOL SERRA DOS AIMORÉS S/A,SERRA DOS AIMORÉS,MG
118,23796998000188,NaN,54.72,57.260,adocao_tardia,COMPANHIA AGRÍCOLA PONTENOVENSE,Urucânia,MG
69,08830263000130,NaN,63.63,63.630,adocao_tardia,Fátima do Sul Agro,Fátima do Sul,MS
33,06312488000179,NaN,61.82,59.745,adocao_tardia,"D'PADUA - DESTILAÇÃO, PRODUÇÃO, AGROINDÚSTRIA ...",Rio Tinto,PB
71,08974214000170,NaN,57.85,57.850,adocao_tardia,COMPANHIA USINA SÃO JOÃO,SANTA RITA,PB


In [9]:
# Auditoria de cancelados/anulados
audit_c = result['audit_cancelados']
print(f'Cancelados audit: {len(audit_c)} CNPJs\n')
print(f'Apenas cancelado/anulado (sem válido): '
      f'{audit_c["apenas_cancelado_anulado"].sum()}')
print(f'Tem válido também: {audit_c["tem_valido_tb"].sum()}')
print()
audit_c.head(10)

Cancelados audit: 283 CNPJs

Apenas cancelado/anulado (sem válido): 2
Tem válido também: 281



,cnpj_clean,statuses,snapshots,razao,cidade,uf,tem_valido_tb,apenas_cancelado_anulado
1,00372496000124,"[cancelado, valido]","[2022, 2025, 2026]",Central Energética Vale do Sapucaí Ltda.,Patrocínio Paulista,SP,True,False
2,00595322000120,"[cancelado, valido]","[2022, 2025, 2026]",DENUSA DESTILARIA NOVA UNIÃO S/A,Jandaia,GO,True,False
3,00738822000255,"[cancelado, valido]","[2022, 2025, 2026]",Santa Cruz Açúcar e Álcool Ltda.,Santa Cruz Cabrália,BA,True,False
4,01105558000102,"[cancelado, valido]","[2022, 2025, 2026]",WD AGROINDUSTRIAL LTDA,João Pinheiro,MG,True,False
5,02126558000143,"[cancelado, valido]","[2022, 2025, 2026]",T.G. AGROINDUSTRIAL LTDA.,Aldeias Altas,MA,True,False
6,02414858000390,"[anulado, cancelado, valido]","[2022, 2025, 2026]",VALE VERDE EMPREENDIMENTOS AGRÍCOLAS LTDA. EM ...,Baia Formosa,RN,True,False
7,02414858000470,"[cancelado, valido]","[2022, 2025, 2026]",Vale Verde Empreendimentos Agrícolas Ltda. Em ...,Itapaci,GO,True,False
8,02460988000105,"[cancelado, valido]","[2025, 2026]",USINA GOIANÉSIA S/A,Goianésia,GO,True,False
9,02635522000195,"[cancelado, valido]","[2022, 2025, 2026]",JALLES MACHADO S.A.,Goianésia,GO,True,False
10,02635522004930,"[cancelado, valido]","[2022, 2025, 2026]",JALLES MACHADO S.A.,Goianésia,GO,True,False


In [10]:
# Auditoria de cidade/UF sem match no crosswalk
audit_u = result['audit_unmatched']
print(f'Unmatched audit: {len(audit_u)} CNPJs\n')
if len(audit_u) > 0:
    cs_ufs = ['SP', 'GO', 'MG', 'PR', 'MS', 'MT']
    audit_u_cs = audit_u[audit_u['uf'].isin(cs_ufs)]
    print(f'Centro-Sul unmatched: {len(audit_u_cs)}')
    print(f'Outras UFs (esperado descartar): {len(audit_u) - len(audit_u_cs)}')
    print(f'\nUFs unmatched:')
    print(audit_u['uf'].value_counts(dropna=False).to_string())
audit_u.head(10)

Unmatched audit: 65 CNPJs

Centro-Sul unmatched: 0
Outras UFs (esperado descartar): 65

UFs unmatched:
uf
None    22
AL      11
PE       9
PB       7
BA       4
RN       3
MA       2
ES       2
PI       1
PA       1
TO       1
SE       1
RJ       1


,cnpj_clean,razao_social,cidade,uf,muni_key,g_data,g_year,muni_key_corrected,geocode,municipio_cw,uf_cw,match_exact,match_method,fuzzy_score,match_any
0,00297598000122,DESTILARIA DE ALCOOL LIBRA LTDA - EM RECUPERAC...,None,None,None|,2025-02-24,2025,None|,NaN,NaN,NaN,False,None,None,False
3,00738822000255,Santa Cruz Açúcar e Álcool Ltda.,Santa Cruz Cabrália,BA,SANTA CRUZ CABRALIA|BA,2020-09-16,2020,SANTA CRUZ CABRALIA|BA,NaN,NaN,NaN,False,None,None,False
5,02126558000143,T.G. AGROINDUSTRIAL LTDA.,Aldeias Altas,MA,ALDEIAS ALTAS|MA,2021-01-22,2021,ALDEIAS ALTAS|MA,NaN,NaN,NaN,False,None,None,False
6,02414858000390,VALE VERDE EMPREENDIMENTOS AGRÍCOLAS LTDA. EM ...,Baia Formosa,RN,BAIA FORMOSA|RN,2021-08-13,2021,BAIA FORMOSA|RN,NaN,NaN,NaN,False,None,None,False
19,03794600000248,ZIHUATANEJO DO BRASIL ACUCAR E ALCOOL S.A EM R...,Rio Formoso,PE,RIO FORMOSO|PE,2022-01-07,2022,RIO FORMOSO|PE,NaN,NaN,NaN,False,None,None,False
26,04588246000187,USINA SANTA MARIA LTDA.,Medeiros Neto,BA,MEDEIROS NETO|BA,2020-10-13,2020,MEDEIROS NETO|BA,NaN,NaN,NaN,False,None,None,False
27,04839268000253,IBERIA INDUSTRIAL E COMERCIAL LTDA - EM RECUPE...,None,None,None|,2024-01-19,2024,None|,NaN,NaN,NaN,False,None,None,False
29,05158542000100,CENTRAL AÇUCAREIRA USINA SANTA MARIA S.A.,Porto Calvo,AL,PORTO CALVO|AL,2020-04-09,2020,PORTO CALVO|AL,NaN,NaN,NaN,False,None,None,False
30,05158542000291,CENTRAL ACUCAREIRA USINA SANTA MARIA S/A.,None,None,None|,2023-07-25,2023,None|,NaN,NaN,NaN,False,None,None,False
32,05343207000182,COMVAP AÇÚCAR E ÁLCOOL LTDA.,União,PI,UNIAO|PI,2020-04-24,2020,UNIAO|PI,NaN,NaN,NaN,False,None,None,False


## Resumo

Se o `muni_treat` salvou ~190+ municípios distribuídos pelas 6 UFs, e as auditorias têm contagens razoáveis, próximo passo:

**`03_seeg.ipynb`** — outcomes AFOLU com Emissão+Remoção e `asinh()` em carbono_solo. Camada mais crítica, com a auditoria F3 (cobertura município × canal × ano).